### **Student Information**
GitHub ID: archaeopteryx36988-pixel

Kaggle name: Fish burger

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**


---

## 1. Model Development (10 pts Required)


**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

* **Merging:** the posts (`final_posts.json`), the train/test split (`data_identification.csv`) and the labels (`emotion.csv`) were merged on `post_id` with `validate="one_to_one"`, giving 3,360 train rows and 1,440 test rows (cell 1-1b).
* **Class imbalance:** the 6 classes are imbalanced (joy 26.0% vs. fear 8.6%, max/min = 3.04, cell 1-2). I kept all rows and handled this with class weights instead: `class_weight="balanced"` for Logistic Regression and a weighted cross-entropy loss for the fine-tuned models.
* **Validation split:** a stratified 80/20 split (`random_state=42`) gives `tr` = 2,688 and `va` = 672 rows with the same class ratios as the full train set (cell 1-4). Every model was scored on this same `va`, so their scores are directly comparable.
* **No text cleaning:** I did not lowercase or remove stopwords, because transformer tokenizers are trained on raw text, and words like "not" can flip the emotion.
* **Length:** token length p95 is 44 (max 512), so `max_length=128` keeps almost every post untruncated while saving GPU time and memory (cell 3-6).
* **Submission format:** `sampleSubmission.csv` uses the column name `post_id`, but Kaggle requires `id`, so every submission renames the column (my first submission failed with "ID column id not found").



### 1.2 Feature Engineering Steps

* **TF-IDF (cell 2-1):** word unigrams + bigrams (`ngram_range=(1, 2)`, `min_df=2`, `sublinear_tf=True`), 6,032 features. Bigrams keep short phrases such as "not happy", where the single words alone would lose the negation. The vectorizer is fit on `tr` only and only transforms `va`/test, so no validation information leaks into the vocabulary or the idf weights.
* **MiniLM sentence embeddings (cell 2-2):** 384-dim vectors from the frozen `all-MiniLM-L6-v2` (the same model as Master Section 5). On their own they score about the same as TF-IDF (macro F1 0.5056 vs. 0.5034), but averaging the two models' probabilities raises it to 0.5421 (cell 3-4), because TF-IDF looks at surface words while MiniLM looks at meaning, so they make different mistakes.
* **Learned features (fine-tuning, cells 3-7 and 3-9b onward):** for MiniLM, roberta-base and deberta-v3-base, I did not hand-design features. The models read the tokenizer output directly and learn task-specific features end-to-end, which gave the largest gain (0.54 → 0.60 → 0.63 validation macro F1).

### 1.3 Explanation of Your Model

**Final model (cell 3-20, `k3_rb4_db4_blend`):** an equal-weight (1/3 each) average of the class probabilities of three components:

1. **roberta-base, 4 seeds** (cells 3-9b, 3-16c): fine-tuned with a class-weighted cross-entropy loss, AdamW (lr 2e-5), batch 16, 10% warmup + linear decay, `max_length=128`; trained on Colab (T4 GPU). The probabilities of seed 42 (refit on all train rows) and seeds 1/2/3 are averaged.
2. **deberta-v3-base, 4 seeds** (cells 3-13b, 3-18b): same recipe with an 8-epoch schedule, keeping the best validation epoch of each seed, then averaging the 4 seeds.
3. **"local" blend** (cells 3-4 to 3-9): 0.5 × fine-tuned MiniLM (CPU) + 0.5 × (0.7 TF-IDF LR + 0.3 MiniLM-embedding LR).

**Why this design:**
* **Seed averaging:** a single fine-tuning run is partly luck. roberta seed 42 reached 0.6314, but seeds 1/2/3 averaged only 0.6144, and the best epoch of the three deberta seeds was 2, 6 and 8. Averaging seeds makes each model more stable (deberta: 0.6138 for one seed → 0.6218 for the 3-seed average).
* **Blending different models:** the three components make different mistakes, so blending beats every single model.
* **Equal weights:** 1/3 each gave the best validation score (0.6487, cell 3-19), and it is the simplest choice, so it does not overfit the weights to `va`.
* **Not adopted:** roberta-large (0.6215) and deberta-v3-large (0.5973) were not better than the base models with only 3,360 training rows, and adding deberta-v3-large to the blend lowered the score (cell 3-16).

**Progress (validation macro F1 / Kaggle public):**

| Submission | Validation | Public |
|---|---|---|
| TF-IDF + LR (baseline) | 0.5034 | 0.5100 |
| TF-IDF LR + MiniLM LR ensemble | 0.5421 | 0.5397 |
| + fine-tuned MiniLM | 0.6022 | 0.6049 |
| + roberta-base + deberta-v3-base | 0.6405 | 0.6227 |
| **seed-averaged roberta + deberta, equal 1/3 (final)** | **0.6487** | **0.6240** |

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude (an AI assistant) as a tutor during the competition. My prompts were in Chinese; they are translated below. For each suggestion I checked the result on my own validation set (`va`) before deciding.

**Adopted**

| My prompt | AI suggestion | My decision and why |
|---|---|---|
| "What other methods can I use to climb the leaderboard?" | Fine-tune pretrained transformers instead of using frozen embeddings; since Kaggle GPU was unavailable for my account, run them on Google Colab (free T4 GPU) and save the class probabilities with `post_id` so they can be blended on my PC. | **Adopted.** Fine-tuned MiniLM raised validation macro F1 from 0.5421 to 0.6022, and roberta-base reached 0.6314 on its own (cells 3-7, 3-9b). |
| "Won't training longer give a higher score?" | Use an 8-epoch schedule, record the validation score after every epoch, and keep the best epoch, because too many epochs overfit. | **Adopted for deberta-v3-base.** Validation rose to 0.6138 at epoch 6 and then dropped (0.6107, 0.6086), confirming that more epochs is not always better (cell 3-13b). |
| "I'm only slightly behind the people above me; can I still improve?" | Run more random seeds, because one fine-tuning run is partly luck; average the seeds, then blend the models. | **Adopted.** roberta seed 42 (0.6314) turned out to be the luckiest run (seeds 1/2/3 averaged 0.6144). With 4 seeds each for roberta and deberta, the equal-weight blend reached the best validation (0.6487) and public score (0.6240) (cells 3-16c to 3-20). |
| "How should I choose my final submissions?" | Trust local validation (672 rows) more than the public leaderboard (~430 rows); use the 2 final selections to spread risk. | **Adopted.** Final selections: `k3_rb4_db4_blend` (best validation and public, seed-averaged) and `k3_rb_db_blend` (a different single-seed recipe). |

**Rejected or tried and not adopted**

| AI suggestion | Why I did not use it |
|---|---|
| Use a ready-made emotion model (`j-hartmann/emotion-english-distilroberta-base`). | Its model page states no license, and it was trained partly on GoEmotions (Reddit). The competition's `post_id`s look like Reddit ids, so it might have already seen test posts (data leakage). |
| Try bigger models (roberta-large, deberta-v3-large). | Tried on Colab, but neither beat the base models (0.6215 and 0.5973 vs. 0.6314 and 0.6138), and adding deberta-v3-large to the blend lowered the score (cell 3-16). With 3,360 training rows, the larger models do not pay off. |
| (My own idea) Keep submitting to Kaggle to push the public score, since 5 submissions per day are allowed. | The AI pointed out that the public leaderboard uses only ~430 rows, so scores move by about ±0.02 from noise. I agreed and only submitted when the validation score improved, to avoid overfitting to the public leaderboard. |

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [65]:
### Add the code related to the preprocessing steps in cells inside this section

In [66]:
# 1-1 Load the posts (final_posts.json) and inspect the first record
from pathlib import Path
import json, pandas as pd

DATA = Path("kaggle_data")   # relative path: the notebook sits in the repo root
with open(DATA / "final_posts.json", encoding="utf-8") as f:
    raw = json.load(f)
print(type(raw), len(raw))
first = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
print(json.dumps(first, ensure_ascii=False, indent=2)[:1000])

<class 'list'> 4800
{
  "post_id": "f8bv5zh",
  "text": "Thought it looked weird as well."
}


In [67]:
# 1-1b Merge posts, train/test split and labels on post_id
posts = pd.DataFrame(raw).astype({"post_id": str})
ids = pd.read_csv(DATA / "data_identification.csv", dtype=str)   # dtype=str: keep ids as text so merge keys match
emo = pd.read_csv(DATA / "emotion.csv", dtype=str)

df = (posts
      .merge(ids, on="post_id", how="inner", validate="one_to_one")   # validate: raise an error on duplicate keys
      .merge(emo, on="post_id", how="left",  validate="one_to_one"))  # left join: keep test rows, which have no label

train = df[df["set"] == "train"].copy()
test  = df[df["set"] == "test"].copy()
print(ids["set"].value_counts())
print(len(df), len(train), len(test))
print("train missing labels:", train["label"].isna().sum(), "| test with labels:", test["label"].notna().sum())

set
train    3360
test     1440
Name: count, dtype: int64
4800 3360 1440
train missing labels: 0 | test with labels: 0


In [68]:
# 1-2 Check whether the 6 labels are balanced
vc = train["label"].value_counts()
print(pd.DataFrame({"n": vc, "pct": (vc / len(train)).round(3)}))
print("max/min =", round(vc.max() / vc.min(), 2))

            n    pct
label               
joy       875  0.260
anger     674  0.201
sadness   628  0.187
disgust   454  0.135
surprise  441  0.131
fear      288  0.086
max/min = 3.04


In [69]:
# 1-3 Check the submission format and that its ids match the test set
sub = pd.read_csv(DATA / "sampleSubmission.csv", dtype=str)
print(sub.columns.tolist(), sub.shape)
print(sub["label"].value_counts())
print("ids match test:", set(sub["post_id"]) == set(test["post_id"]))

['post_id', 'label'] (1440, 2)
label
sadness    1440
Name: count, dtype: int64
ids match test: True


In [70]:
# 1-4 Hold out 20% of train for validation (stratify keeps class ratios; random_state for reproducibility)
from sklearn.model_selection import train_test_split
tr, va = train_test_split(train, test_size=0.2, stratify=train["label"], random_state=42)
print(len(tr), len(va))
print(va["label"].value_counts())

2688 672
label
joy         175
anger       135
sadness     126
disgust      91
surprise     88
fear         57
Name: count, dtype: int64


## 2. Feature Engineering Steps

In [71]:
### Add the code related to the feature engineering steps in cells inside this section

In [72]:
# 2-1 TF-IDF: fit on tr only; va/test are only transformed, so no validation info leaks
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_tr = vec.fit_transform(tr["text"])
X_va = vec.transform(va["text"])
print(X_tr.shape, X_va.shape)

(2688, 6032) (672, 6032)


In [73]:
# 2-2 MiniLM sentence embeddings (same all-MiniLM-L6-v2 as Master Section 5; frozen weights, so no leakage)
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("all-MiniLM-L6-v2")
E_tr = st.encode(tr["text"].tolist(), batch_size=64, show_progress_bar=True)
E_va = st.encode(va["text"].tolist(), batch_size=64, show_progress_bar=True)
E_te = st.encode(test["text"].tolist(), batch_size=64, show_progress_bar=True)
print(E_tr.shape, E_va.shape, E_te.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

(2688, 384) (672, 384) (1440, 384)


## 3. Model Implementation Steps

In [74]:
### Add the code related to the model implementation steps in cells inside this section

In [75]:
# 3-1 Baseline: Logistic Regression + class_weight="balanced"; report both macro F1 and accuracy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_tr, tr["label"])
pred_va = clf.predict(X_va)
print("macro F1:", round(f1_score(va["label"], pred_va, average="macro"), 4),
      "| accuracy:", round(accuracy_score(va["label"], pred_va), 4))
print(classification_report(va["label"], pred_va, digits=3))

macro F1: 0.5034 | accuracy: 0.5238
              precision    recall  f1-score   support

       anger      0.484     0.437     0.459       135
     disgust      0.368     0.385     0.376        91
        fear      0.500     0.421     0.457        57
         joy      0.635     0.646     0.640       175
     sadness      0.559     0.563     0.561       126
    surprise      0.490     0.568     0.526        88

    accuracy                          0.524       672
   macro avg      0.506     0.503     0.503       672
weighted avg      0.524     0.524     0.523       672



In [76]:
# 3-2 Refit on all 3,360 train rows (same settings as 3-1), predict test, write the submission file
vec_full = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_full = vec_full.fit_transform(train["text"])
X_te = vec_full.transform(test["text"])
clf_full = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_full, train["label"])

out = pd.DataFrame({"post_id": test["post_id"], "label": clf_full.predict(X_te)})
out = sub[["post_id"]].merge(out, on="post_id", how="left")      # keep sampleSubmission's row order
assert len(out) == 1440 and out["label"].notna().all()
Path("submissions").mkdir(exist_ok=True)
out.rename(columns={"post_id": "id"}).to_csv("submissions/k2_tfidf_lr.csv", index=False)   # Kaggle requires the column name "id"
print(out.shape, out["label"].value_counts().to_dict())

(1440, 2) {'joy': 339, 'sadness': 305, 'anger': 302, 'surprise': 192, 'disgust': 189, 'fear': 113}


In [77]:
# 3-3 MiniLM embeddings + LR; tune C (inverse regularization strength) on the validation set
for C in [0.5, 1, 2, 5, 10]:
    m = LogisticRegression(max_iter=2000, class_weight="balanced", C=C).fit(E_tr, tr["label"])
    p = m.predict(E_va)
    print(f"C={C:<4} macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

C=0.5  macro F1=0.5056  acc=0.5268
C=1    macro F1=0.4978  acc=0.5193
C=2    macro F1=0.4944  acc=0.5164
C=5    macro F1=0.4951  acc=0.5149
C=10   macro F1=0.4858  acc=0.5060


In [78]:
# 3-4 Soft-voting ensemble: average class probabilities of the TF-IDF LR (3-1) and the MiniLM LR (C=0.5)
m_emb = LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5).fit(E_tr, tr["label"])
for w in [0.3, 0.5, 0.7]:
    proba = w * clf.predict_proba(X_va) + (1 - w) * m_emb.predict_proba(E_va)
    p = clf.classes_[proba.argmax(axis=1)]
    print(f"w_tfidf={w} macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

w_tfidf=0.3 macro F1=0.5256  acc=0.5461
w_tfidf=0.5 macro F1=0.5382  acc=0.5580
w_tfidf=0.7 macro F1=0.5421  acc=0.5625


In [79]:
# 3-5 Refit both models on all train rows, ensemble with w_tfidf=0.7, write the submission file
E_full = st.encode(train["text"].tolist(), batch_size=64, show_progress_bar=True)
m_emb_full = LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5).fit(E_full, train["label"])
proba_te = 0.7 * clf_full.predict_proba(X_te) + 0.3 * m_emb_full.predict_proba(E_te)   # clf_full / X_te come from 3-2

out = pd.DataFrame({"post_id": test["post_id"], "label": clf_full.classes_[proba_te.argmax(axis=1)]})
out = sub[["post_id"]].merge(out, on="post_id", how="left")      # keep sampleSubmission's row order
assert len(out) == 1440 and out["label"].notna().all()
out.rename(columns={"post_id": "id"}).to_csv("submissions/k3_ens_tfidf_minilm.csv", index=False)
print(out.shape, out["label"].value_counts().to_dict())

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

(1440, 2) {'joy': 358, 'anger': 310, 'sadness': 267, 'surprise': 211, 'disgust': 182, 'fear': 112}


In [80]:
# 3-6 Fine-tune MiniLM on CPU, step 1: label ids, tokenizer, and token-length check
import torch, numpy as np
from transformers import AutoTokenizer
torch.manual_seed(42)                                   # fixed seed for reproducibility
labels = sorted(train["label"].unique())
lab2id = {l: i for i, l in enumerate(labels)}
MODEL = "sentence-transformers/all-MiniLM-L6-v2"        # same model as 2-2, already downloaded
tok = AutoTokenizer.from_pretrained(MODEL)
lens = [len(ids) for ids in tok(train["text"].tolist())["input_ids"]]
print(labels)
print("token length p95 =", int(np.percentile(lens, 95)), "| max =", max(lens))

['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
token length p95 = 44 | max = 512


In [81]:
# 3-7 Fine-tune MiniLM on CPU, step 2: train on tr, report val macro F1 after each epoch
import time
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

def make_loader(df, shuffle):
    enc = tok(df["text"].tolist(), truncation=True, max_length=128)    # covers >95% of texts untruncated
    items = [{"input_ids": a, "attention_mask": m} for a, m in zip(enc["input_ids"], enc["attention_mask"])]
    if df["label"].notna().all():                                      # test rows have no label
        for it, l in zip(items, df["label"]): it["labels"] = lab2id[l]
    return DataLoader(items, batch_size=32, shuffle=shuffle, collate_fn=DataCollatorWithPadding(tok))

def predict_proba(model, dl):
    model.eval(); out = []
    with torch.no_grad():
        for b in dl:
            b.pop("labels", None)
            out.append(torch.softmax(model(**b).logits, -1))
    return torch.cat(out).numpy()

torch.manual_seed(42)
counts = tr["label"].value_counts()
cw = torch.tensor([len(tr) / (len(labels) * counts[l]) for l in labels], dtype=torch.float)
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)                         # same idea as class_weight="balanced"
ft = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels))
opt = torch.optim.AdamW(ft.parameters(), lr=5e-5, weight_decay=0.01)
dl_tr, dl_va = make_loader(tr, True), make_loader(va, False)

for epoch in range(5):
    ft.train(); t0 = time.time()
    for b in dl_tr:
        y = b.pop("labels")
        loss = loss_fn(ft(**b).logits, y)
        loss.backward(); opt.step(); opt.zero_grad()
    p = np.array(labels)[predict_proba(ft, dl_va).argmax(1)]
    print(f"epoch {epoch+1}: macro F1={f1_score(va['label'], p, average='macro'):.4f}  "
          f"acc={accuracy_score(va['label'], p):.4f}  ({time.time()-t0:.0f}s)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: macro F1=0.5655  acc=0.5923  (102s)
epoch 2: macro F1=0.5941  acc=0.6190  (99s)
epoch 3: macro F1=0.5940  acc=0.6235  (101s)
epoch 4: macro F1=0.5787  acc=0.5997  (113s)
epoch 5: macro F1=0.5957  acc=0.6235  (111s)


In [82]:
# 3-8 Validation only: fine-tuned model alone vs. blended with the 3-4 ensemble
P_ft  = predict_proba(ft, dl_va)
P_ens = 0.7 * clf.predict_proba(X_va) + 0.3 * m_emb.predict_proba(E_va)   # 3-4 ensemble; class order = labels (alphabetical)
for w in [1.0, 0.7, 0.5]:
    p = np.array(labels)[(w * P_ft + (1 - w) * P_ens).argmax(1)]
    print(f"w_ft={w}  macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

w_ft=1.0  macro F1=0.5957  acc=0.6235
w_ft=0.7  macro F1=0.5999  acc=0.6265
w_ft=0.5  macro F1=0.6022  acc=0.6280


In [83]:
# 3-9 Refit the fine-tuned model on all train rows (3 epochs), predict test, write two submission files
torch.manual_seed(42)
cw_full = torch.tensor([len(train) / (len(labels) * train["label"].value_counts()[l]) for l in labels], dtype=torch.float)
loss_full = torch.nn.CrossEntropyLoss(weight=cw_full)
ft_full = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels))
opt = torch.optim.AdamW(ft_full.parameters(), lr=5e-5, weight_decay=0.01)
dl_full, dl_te = make_loader(train, True), make_loader(test, False)
for epoch in range(3):                                    # val F1 plateaued after epoch 2-3 in 3-7
    ft_full.train(); t0 = time.time()
    for b in dl_full:
        y = b.pop("labels")
        loss_full(ft_full(**b).logits, y).backward(); opt.step(); opt.zero_grad()
    print(f"epoch {epoch+1} done ({time.time()-t0:.0f}s)")

P_ft_te = predict_proba(ft_full, dl_te)
for name, P in [("k3_ft_minilm", P_ft_te), ("k3_ft_blend", 0.5 * P_ft_te + 0.5 * proba_te)]:   # proba_te: 3-5 ensemble
    out = pd.DataFrame({"post_id": test["post_id"].values, "label": np.array(labels)[P.argmax(1)]})
    out = sub[["post_id"]].merge(out, on="post_id", how="left")      # keep sampleSubmission's row order
    assert len(out) == 1440 and out["label"].notna().all()
    out.rename(columns={"post_id": "id"}).to_csv(f"submissions/{name}.csv", index=False)
    print(name, out["label"].value_counts().to_dict())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1 done (128s)
epoch 2 done (136s)
epoch 3 done (134s)
k3_ft_minilm {'anger': 373, 'joy': 339, 'disgust': 212, 'sadness': 202, 'surprise': 198, 'fear': 116}
k3_ft_blend {'anger': 369, 'joy': 342, 'sadness': 206, 'disgust': 206, 'surprise': 202, 'fear': 115}


**3-9b roberta-base fine-tuning (run on Google Colab, T4 GPU)**

roberta-base (~125M parameters) is too slow to fine-tune on this PC's CPU, so the next cell was run on Google Colab with a free T4 GPU,
using the same data, the same merge, and the same stratified split (`random_state=42`), so its validation scores are directly comparable.
It saved the class probabilities to `submissions/rb_proba_va.csv` and `submissions/rb_proba_te.csv`, which cells 3-10 and 3-11 read.
The cell is guarded by `torch.cuda.is_available()`, so running this notebook on a CPU skips it.

Colab results (validation macro F1 per epoch): 0.5617, 0.5782, 0.6314, 0.6290 (acc 0.6577); about 33 s per epoch.

In [84]:
# 3-9b roberta-base fine-tuning: run on Google Colab (T4 GPU); skipped on CPU.
# Writes submissions/k3_roberta.csv and submissions/rb_proba_va.csv / rb_proba_te.csv (read by 3-10 and 3-11)
import torch
if torch.cuda.is_available():
    import time
    from torch.utils.data import DataLoader
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              DataCollatorWithPadding, get_linear_schedule_with_warmup)
    dev, RB = "cuda", "roberta-base"                      # roberta-base: MIT license, ~125M params
    rb_tok = AutoTokenizer.from_pretrained(RB)

    def rb_loader(d, shuffle):
        enc = rb_tok(d["text"].tolist(), truncation=True, max_length=128)
        items = [{"input_ids": a, "attention_mask": m} for a, m in zip(enc["input_ids"], enc["attention_mask"])]
        if d["label"].notna().all():                     # test rows have no label
            for it, l in zip(items, d["label"]): it["labels"] = lab2id[l]
        return DataLoader(items, batch_size=16, shuffle=shuffle, collate_fn=DataCollatorWithPadding(rb_tok))

    def rb_proba(model, dl):
        model.eval(); out = []
        with torch.no_grad():
            for b in dl:
                b.pop("labels", None)
                out.append(torch.softmax(model(**{k: v.to(dev) for k, v in b.items()}).logits, -1).cpu())
        return torch.cat(out).numpy()

    def rb_train(d, epochs, eval_dl=None, seed=42):
        torch.manual_seed(seed)
        cnt = d["label"].value_counts()
        cw = torch.tensor([len(d) / (len(labels) * cnt[l]) for l in labels], dtype=torch.float, device=dev)
        loss_fn = torch.nn.CrossEntropyLoss(weight=cw)   # same idea as class_weight="balanced"
        model = AutoModelForSequenceClassification.from_pretrained(RB, num_labels=len(labels)).to(dev)
        dl = rb_loader(d, True)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * epochs * len(dl)), epochs * len(dl))   # warmup stabilizes roberta
        for ep in range(epochs):
            model.train(); t0 = time.time()
            for b in dl:
                b = {k: v.to(dev) for k, v in b.items()}; y = b.pop("labels")
                loss_fn(model(**b).logits, y).backward(); opt.step(); sch.step(); opt.zero_grad()
            if eval_dl is not None:
                p = np.array(labels)[rb_proba(model, eval_dl).argmax(1)]
                print(f"epoch {ep+1}: macro F1={f1_score(va['label'], p, average='macro'):.4f}  "
                      f"acc={accuracy_score(va['label'], p):.4f}  ({time.time()-t0:.0f}s)")
        return model

    dl_va_rb = rb_loader(va, False)
    model_va = rb_train(tr, 4, dl_va_rb)                 # validation run
    P_va_rb = rb_proba(model_va, dl_va_rb)
    del model_va; torch.cuda.empty_cache()
    model_full = rb_train(train, 4)                      # refit on all 3,360 train rows
    P_te_rb = rb_proba(model_full, rb_loader(test, False))

    out = pd.DataFrame({"post_id": test["post_id"].values, "label": np.array(labels)[P_te_rb.argmax(1)]})
    out = sub[["post_id"]].merge(out, on="post_id", how="left")
    out.rename(columns={"post_id": "id"}).to_csv("submissions/k3_roberta.csv", index=False)
    pd.DataFrame(P_va_rb, columns=labels).assign(post_id=va["post_id"].values).to_csv("submissions/rb_proba_va.csv", index=False)
    pd.DataFrame(P_te_rb, columns=labels).assign(post_id=test["post_id"].values).to_csv("submissions/rb_proba_te.csv", index=False)
else:
    print("No GPU: skipped. Using the saved Colab outputs in submissions/rb_proba_va.csv and rb_proba_te.csv")

No GPU: skipped. Using the saved Colab outputs in submissions/rb_proba_va.csv and rb_proba_te.csv


In [85]:
# 3-10 Validation only: blend the Colab roberta-base probabilities with the local k3_ft_blend recipe (3-8)
rb_va = (pd.read_csv("submissions/rb_proba_va.csv", dtype={"post_id": str})
           .set_index("post_id").loc[va["post_id"], labels].values)          # align rows to va by post_id
P_local = 0.5 * P_ft + 0.5 * P_ens                                           # same recipe as k3_ft_blend
for w in [1.0, 0.7, 0.5, 0.3]:
    p = np.array(labels)[(w * rb_va + (1 - w) * P_local).argmax(1)]
    print(f"w_roberta={w}  macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

w_roberta=1.0  macro F1=0.6299  acc=0.6577
w_roberta=0.7  macro F1=0.6298  acc=0.6562
w_roberta=0.5  macro F1=0.6368  acc=0.6637
w_roberta=0.3  macro F1=0.6214  acc=0.6458


In [86]:
# 3-11 Final blend for test: 0.5 roberta-base (Colab) + 0.5 k3_ft_blend recipe (3-9), write the submission file
rb_te = (pd.read_csv("submissions/rb_proba_te.csv", dtype={"post_id": str})
           .set_index("post_id").loc[test["post_id"], labels].values)        # align rows to test by post_id
P_te_final = 0.5 * rb_te + 0.5 * (0.5 * P_ft_te + 0.5 * proba_te)            # P_ft_te from 3-9, proba_te from 3-5
out = pd.DataFrame({"post_id": test["post_id"].values, "label": np.array(labels)[P_te_final.argmax(1)]})
out = sub[["post_id"]].merge(out, on="post_id", how="left")                  # keep sampleSubmission's row order
assert len(out) == 1440 and out["label"].notna().all()
out.rename(columns={"post_id": "id"}).to_csv("submissions/k3_rb_blend.csv", index=False)
print(out["label"].value_counts().to_dict())

{'joy': 335, 'anger': 311, 'sadness': 267, 'surprise': 207, 'disgust': 194, 'fear': 126}


**3-11b deberta-v3-base fine-tuning (run on Google Colab, T4 GPU)**

Same data, merge, split (`random_state=42`) and recipe as 3-9b (weighted CE, AdamW lr 2e-5, batch 16, 10% warmup + linear decay, seed 42).
The Colab code is rewritten here as one reusable function `cf_run`, which the later Colab cells (3-13b, 3-16b, 3-17b, 3-19b) also use.
deberta needs `logits.float()` and `labels.long()`, otherwise `cross_entropy` raises a dtype error.
Guarded by `torch.cuda.is_available()`, so it is skipped on CPU; the outputs are `submissions/db_proba_va.csv` and `db_proba_te.csv`.

Colab results (validation macro F1 per epoch): 0.4236, 0.5417, 0.5893, 0.5946 (still rising, so 4 epochs is undertrained; see 3-13b).

In [87]:
# 3-11b deberta-v3-base fine-tuning: run on Google Colab (T4 GPU); skipped on CPU.
# Defines cf_run (reused by later Colab cells) and writes submissions/db_proba_va.csv / db_proba_te.csv
if torch.cuda.is_available():
    from transformers import get_linear_schedule_with_warmup
    dev = "cuda"

    def cf_run(name, d, epochs, pred_sets, lr=2e-5, bs=16, seed=42):
        """Fine-tune `name` on d; return {key: [probabilities after each epoch]} for every DataFrame in pred_sets."""
        torch.manual_seed(seed)
        ctok = AutoTokenizer.from_pretrained(name)
        def loader(x, shuffle):
            enc = ctok(x["text"].tolist(), truncation=True, max_length=128)
            items = [{"input_ids": a, "attention_mask": m} for a, m in zip(enc["input_ids"], enc["attention_mask"])]
            if x["label"].notna().all():                                   # test rows have no label
                for it, l in zip(items, x["label"]): it["labels"] = lab2id[l]
            return DataLoader(items, batch_size=bs, shuffle=shuffle, collate_fn=DataCollatorWithPadding(ctok))
        cnt = d["label"].value_counts()
        cw = torch.tensor([len(d) / (len(labels) * cnt[l]) for l in labels], dtype=torch.float, device=dev)
        loss_fn = torch.nn.CrossEntropyLoss(weight=cw)                     # same idea as class_weight="balanced"
        model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=len(labels)).to(dev)
        dl, dls = loader(d, True), {k: loader(v, False) for k, v in pred_sets.items()}
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * epochs * len(dl)), epochs * len(dl))
        hist = {k: [] for k in pred_sets}
        for ep in range(epochs):
            model.train()
            for b in dl:
                b = {k: v.to(dev) for k, v in b.items()}; y = b.pop("labels").long()
                loss_fn(model(**b).logits.float(), y).backward(); opt.step(); sch.step(); opt.zero_grad()
            model.eval()
            with torch.no_grad():                                          # probabilities after every epoch
                for k, dlk in dls.items():
                    P = [torch.softmax(model(**{n: v.to(dev) for n, v in b.items() if n != "labels"}).logits.float(), -1).cpu()
                         for b in dlk]
                    hist[k].append(torch.cat(P).numpy())
            if "va" in hist:
                p = np.array(labels)[hist["va"][-1].argmax(1)]
                print(f"{name} seed {seed} epoch {ep+1}: macro F1={f1_score(va['label'], p, average='macro'):.4f}")
        del model; torch.cuda.empty_cache()
        return hist

    def save_proba(P, d, path):                                            # keep post_id so later cells can align rows
        pd.DataFrame(P, columns=labels).assign(post_id=d["post_id"].values).to_csv(path, index=False)

    DB = "microsoft/deberta-v3-base"                                       # MIT license, ~184M params
    h_va = cf_run(DB, tr, 4, {"va": va})                                   # validation run on tr
    h_te = cf_run(DB, train, 4, {"te": test})                              # refit on all 3,360 train rows
    save_proba(h_va["va"][-1], va, "submissions/db_proba_va.csv")
    save_proba(h_te["te"][-1], test, "submissions/db_proba_te.csv")
else:
    print("No GPU: skipped. Using the saved Colab outputs in submissions/db_proba_va.csv and db_proba_te.csv")

No GPU: skipped. Using the saved Colab outputs in submissions/db_proba_va.csv and db_proba_te.csv


In [88]:
# 3-12 Validation only: does adding deberta-v3-base (Colab) to the 3-10 blend help?
db_va = (pd.read_csv("submissions/db_proba_va.csv", dtype={"post_id": str})
           .set_index("post_id").loc[va["post_id"], labels].values)        # align rows to va by post_id
for name, P in [("rb 0.5 + local 0.5 (3-10 best)", 0.5 * rb_va + 0.5 * P_local),
                ("rb + db + local, equal 1/3",     (rb_va + db_va + P_local) / 3),
                ("rb 0.4 + db 0.2 + local 0.4",    0.4 * rb_va + 0.2 * db_va + 0.4 * P_local)]:
    p = np.array(labels)[P.argmax(1)]
    print(f"{name:32s} macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

rb 0.5 + local 0.5 (3-10 best)   macro F1=0.6368  acc=0.6637
rb + db + local, equal 1/3       macro F1=0.6260  acc=0.6548
rb 0.4 + db 0.2 + local 0.4      macro F1=0.6405  acc=0.6667


In [89]:
# 3-13 Test blend: 0.4 roberta-base + 0.2 deberta-v3-base + 0.4 k3_ft_blend recipe, write the submission file
db_te = (pd.read_csv("submissions/db_proba_te.csv", dtype={"post_id": str})
           .set_index("post_id").loc[test["post_id"], labels].values)        # align rows to test by post_id
P_te3 = 0.4 * rb_te + 0.2 * db_te + 0.4 * (0.5 * P_ft_te + 0.5 * proba_te)  # rb_te from 3-11, P_ft_te 3-9, proba_te 3-5
out = pd.DataFrame({"post_id": test["post_id"].values, "label": np.array(labels)[P_te3.argmax(1)]})
out = sub[["post_id"]].merge(out, on="post_id", how="left")                  # keep sampleSubmission's row order
assert len(out) == 1440 and out["label"].notna().all()
out.rename(columns={"post_id": "id"}).to_csv("submissions/k3_rb_db_blend.csv", index=False)
print(out["label"].value_counts().to_dict())

{'joy': 340, 'anger': 304, 'sadness': 272, 'surprise': 210, 'disgust': 184, 'fear': 130}


**3-13b deberta-v3-base with a longer schedule (run on Google Colab, T4 GPU)**

4 epochs in 3-11b were still improving, so deberta was trained with an 8-epoch schedule and the validation probabilities were kept after every epoch.
The best validation epoch (6) was then used for the full-data refit with the same 8-epoch schedule.
Uses `cf_run` from 3-11b; outputs `submissions/db8_proba_va.csv` and `db8_proba_te.csv` (read by 3-14 and 3-15).

Colab results (validation macro F1 per epoch): 0.2653, 0.5651, 0.5886, 0.6002, 0.6039, **0.6138**, 0.6107, 0.6086; about 52 s per epoch.
Epochs 7-8 drop again, so training longer starts to overfit. Epoch 6 was chosen on the validation set, so its validation score is slightly optimistic.

In [90]:
# 3-13b deberta-v3-base, 8-epoch schedule, keep epoch 6: run on Google Colab (T4 GPU); skipped on CPU.
if torch.cuda.is_available():
    h_va = cf_run(DB, tr, 8, {"va": va})                       # validation run: probabilities after every epoch
    h_te = cf_run(DB, train, 8, {"te": test})                  # full-data refit with the same schedule
    BEST = 6                                                   # best validation epoch above
    save_proba(h_va["va"][BEST - 1], va, "submissions/db8_proba_va.csv")
    save_proba(h_te["te"][BEST - 1], test, "submissions/db8_proba_te.csv")
else:
    print("No GPU: skipped. Using the saved Colab outputs in submissions/db8_proba_va.csv and db8_proba_te.csv")

No GPU: skipped. Using the saved Colab outputs in submissions/db8_proba_va.csv and db8_proba_te.csv


In [91]:
# 3-14 Validation only: blends with the longer-trained deberta (8-epoch schedule, best epoch 6)
db8_va = (pd.read_csv("submissions/db8_proba_va.csv", dtype={"post_id": str})
            .set_index("post_id").loc[va["post_id"], labels].values)       # align rows to va by post_id
for name, P in [("rb .4 + db .2 + local .4 (submitted)", 0.4 * rb_va + 0.2 * db_va + 0.4 * P_local),
                ("rb .4 + db8 .2 + local .4",           0.4 * rb_va + 0.2 * db8_va + 0.4 * P_local),
                ("rb .35 + db8 .3 + local .35",         0.35 * rb_va + 0.3 * db8_va + 0.35 * P_local),
                ("rb + db8 + local, equal 1/3",         (rb_va + db8_va + P_local) / 3)]:
    p = np.array(labels)[P.argmax(1)]
    print(f"{name:38s} macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

rb .4 + db .2 + local .4 (submitted)   macro F1=0.6405  acc=0.6667
rb .4 + db8 .2 + local .4              macro F1=0.6458  acc=0.6711
rb .35 + db8 .3 + local .35            macro F1=0.6361  acc=0.6622
rb + db8 + local, equal 1/3            macro F1=0.6344  acc=0.6622


In [92]:
# 3-15 Test blend with the longer-trained deberta: 0.4 roberta + 0.2 deberta(8-ep schedule, epoch 6) + 0.4 local
db8_te = (pd.read_csv("submissions/db8_proba_te.csv", dtype={"post_id": str})
            .set_index("post_id").loc[test["post_id"], labels].values)
P_te4 = 0.4 * rb_te + 0.2 * db8_te + 0.4 * (0.5 * P_ft_te + 0.5 * proba_te)
out = pd.DataFrame({"post_id": test["post_id"].values, "label": np.array(labels)[P_te4.argmax(1)]})
out = sub[["post_id"]].merge(out, on="post_id", how="left")
assert len(out) == 1440 and out["label"].notna().all()
out.rename(columns={"post_id": "id"}).to_csv("submissions/k3_rb_db8_blend.csv", index=False)
print(out["label"].value_counts().to_dict())

{'joy': 347, 'anger': 304, 'sadness': 268, 'surprise': 206, 'disgust': 192, 'fear': 123}


**3-16b deberta-v3-large (run on Google Colab, T4 GPU) — tried, not adopted**

To test whether a bigger model helps, deberta-v3-large (MIT license, ~435M params) was fine-tuned with lr 1e-5, batch 8 and an 8-epoch schedule.
To save GPU time it was trained once on `tr` only (no full-data refit); validation and test probabilities were kept after every epoch, and epoch 7 (best on validation) was saved.
Uses `cf_run` from 3-11b; outputs `submissions/dbL_proba_va.csv` and `dbL_proba_te.csv` (read by 3-16).

Colab results (validation macro F1 per epoch): 0.2626, 0.5455, 0.5597, 0.5916, 0.5883, 0.5914, **0.5973**, 0.5970; about 176 s per epoch.
It is weaker than deberta-v3-base (0.6138), and 3-16 shows that adding it to the blend does not help, so it was not used. With only 3,360 training rows, the larger model does not pay off.

In [93]:
# 3-16b deberta-v3-large (tried, not adopted): run on Google Colab (T4 GPU); skipped on CPU.
if torch.cuda.is_available():
    DBL = "microsoft/deberta-v3-large"                                     # MIT license, ~435M params
    h = cf_run(DBL, tr, 8, {"va": va, "te": test}, lr=1e-5, bs=8)          # one run on tr, keep va/test after every epoch
    BEST = 7                                                               # best validation epoch above
    save_proba(h["va"][BEST - 1], va, "submissions/dbL_proba_va.csv")
    save_proba(h["te"][BEST - 1], test, "submissions/dbL_proba_te.csv")
else:
    print("No GPU: skipped. Using the saved Colab outputs in submissions/dbL_proba_va.csv and dbL_proba_te.csv")

No GPU: skipped. Using the saved Colab outputs in submissions/dbL_proba_va.csv and dbL_proba_te.csv


In [94]:
# 3-16 Validation only: does adding deberta-v3-large (Colab, trained on tr only) to the best blend help?
dbL_va = (pd.read_csv("submissions/dbL_proba_va.csv", dtype={"post_id": str})
            .set_index("post_id").loc[va["post_id"], labels].values)
base = 0.4 * rb_va + 0.2 * db8_va + 0.4 * P_local                             # = k3_rb_db8_blend (0.6458)
for w in [0.0, 0.1, 0.2]:
    p = np.array(labels)[((1 - w) * base + w * dbL_va).argmax(1)]
    print(f"w_large={w}  macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

w_large=0.0  macro F1=0.6458  acc=0.6711
w_large=0.1  macro F1=0.6387  acc=0.6637
w_large=0.2  macro F1=0.6413  acc=0.6667


**3-16c roberta-base with 3 more random seeds (run on Google Colab, T4 GPU)**

To check how much of roberta's score is luck, roberta-base was fine-tuned again with seeds 1, 2 and 3 (same recipe as 3-9b, 4 epochs, trained on `tr` only).
For each seed the best validation epoch was kept, and the validation/test probabilities of the 3 seeds were averaged.
Uses `cf_run` from 3-11b; outputs `submissions/rbS_proba_va.csv` and `rbS_proba_te.csv` (read by 3-17 and 3-18).

Colab results (validation macro F1 per epoch):
* seed 1: 0.5592, 0.6147, 0.6146, 0.6123
* seed 2: 0.5813, 0.5600, 0.6083, 0.5996
* seed 3: 0.5722, 0.6109, 0.6177, 0.6069

3-seed average: 0.6144. Seed 42 (0.6314 in 3-9b) was the luckiest run; changing only the seed moves the score by about 0.02.

In [95]:
# 3-16c roberta-base, seeds 1/2/3, best epoch per seed, averaged: run on Google Colab (T4 GPU); skipped on CPU.
if torch.cuda.is_available():
    Pva, Pte = [], []
    for s in [1, 2, 3]:
        h = cf_run("roberta-base", tr, 4, {"va": va, "te": test}, seed=s)
        f1s = [f1_score(va["label"], np.array(labels)[P.argmax(1)], average="macro") for P in h["va"]]
        b = int(np.argmax(f1s))                                            # best validation epoch for this seed
        Pva.append(h["va"][b]); Pte.append(h["te"][b])
    save_proba(np.mean(Pva, 0), va, "submissions/rbS_proba_va.csv")       # equal-weight average over seeds
    save_proba(np.mean(Pte, 0), test, "submissions/rbS_proba_te.csv")
else:
    print("No GPU: skipped. Using the saved Colab outputs in submissions/rbS_proba_va.csv and rbS_proba_te.csv")

No GPU: skipped. Using the saved Colab outputs in submissions/rbS_proba_va.csv and rbS_proba_te.csv


In [96]:
# 3-17 Validation only: replace the single roberta (seed 42) by the average of 4 seeds (42 + 1, 2, 3)
rbS_va = (pd.read_csv("submissions/rbS_proba_va.csv", dtype={"post_id": str})
            .set_index("post_id").loc[va["post_id"], labels].values)
rb4_va = (rb_va + 3 * rbS_va) / 4                                            # equal weight per seed
for name, P in [("rb(42) .4 + db8 .2 + local .4 (submitted)", 0.4 * rb_va  + 0.2 * db8_va + 0.4 * P_local),
                ("rb(4 seeds) .4 + db8 .2 + local .4",       0.4 * rb4_va + 0.2 * db8_va + 0.4 * P_local),
                ("rb(4 seeds) .5 + db8 .2 + local .3",       0.5 * rb4_va + 0.2 * db8_va + 0.3 * P_local)]:
    p = np.array(labels)[P.argmax(1)]
    print(f"{name:42s} macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

rb(42) .4 + db8 .2 + local .4 (submitted)  macro F1=0.6458  acc=0.6711
rb(4 seeds) .4 + db8 .2 + local .4         macro F1=0.6434  acc=0.6667
rb(4 seeds) .5 + db8 .2 + local .3         macro F1=0.6390  acc=0.6652


In [97]:
# 3-18 Test blend: roberta averaged over 4 seeds .4 + deberta(8-ep sched, ep6) .2 + local .4, write the submission file
rbS_te = (pd.read_csv("submissions/rbS_proba_te.csv", dtype={"post_id": str})
            .set_index("post_id").loc[test["post_id"], labels].values)
rb4_te = (rb_te + 3 * rbS_te) / 4                                            # seed 42 (full-data refit) + seeds 1, 2, 3
P_te5 = 0.4 * rb4_te + 0.2 * db8_te + 0.4 * (0.5 * P_ft_te + 0.5 * proba_te)
out = pd.DataFrame({"post_id": test["post_id"].values, "label": np.array(labels)[P_te5.argmax(1)]})
out = sub[["post_id"]].merge(out, on="post_id", how="left")
assert len(out) == 1440 and out["label"].notna().all()
out.rename(columns={"post_id": "id"}).to_csv("submissions/k3_rb4_db8_blend.csv", index=False)
print(out["label"].value_counts().to_dict())

{'joy': 357, 'anger': 340, 'sadness': 267, 'surprise': 203, 'disgust': 157, 'fear': 116}


**3-18b deberta-v3-base with 3 more random seeds (run on Google Colab, T4 GPU)**

Same idea as 3-16c: deberta-v3-base (8-epoch schedule as in 3-13b) was fine-tuned again with seeds 1, 2 and 3, trained on `tr` only.
For each seed the best validation epoch was kept, and the validation/test probabilities of the 3 seeds were averaged.
Uses `cf_run` from 3-11b; outputs `submissions/dbS_proba_va.csv` and `dbS_proba_te.csv` (read by 3-19 and 3-20).

Colab results (validation macro F1):
* seed 1: 0.1535, 0.6171, 0.6145, 0.5950, 0.6032, 0.6156, 0.6133, 0.6140 (best epoch 2)
* seed 2: 0.2943, 0.5143, 0.6087, 0.6078, 0.5903, 0.6151, 0.5995, 0.6065 (best epoch 6)
* seed 3: 0.3321, 0.5651, 0.5730, 0.5978, 0.5912, 0.5950, 0.5991, 0.5998 (best epoch 8)

3-seed average: 0.6218, higher than the single seed-42 run (0.6138), so averaging seeds makes deberta both stronger and more stable.

In [98]:
# 3-18b deberta-v3-base, seeds 1/2/3, 8-epoch schedule, best epoch per seed, averaged: run on Google Colab (T4 GPU); skipped on CPU.
if torch.cuda.is_available():
    Pva, Pte = [], []
    for s in [1, 2, 3]:
        h = cf_run(DB, tr, 8, {"va": va, "te": test}, seed=s)
        f1s = [f1_score(va["label"], np.array(labels)[P.argmax(1)], average="macro") for P in h["va"]]
        b = int(np.argmax(f1s))                                            # best validation epoch for this seed
        Pva.append(h["va"][b]); Pte.append(h["te"][b])
    save_proba(np.mean(Pva, 0), va, "submissions/dbS_proba_va.csv")       # equal-weight average over seeds
    save_proba(np.mean(Pte, 0), test, "submissions/dbS_proba_te.csv")
else:
    print("No GPU: skipped. Using the saved Colab outputs in submissions/dbS_proba_va.csv and dbS_proba_te.csv")

No GPU: skipped. Using the saved Colab outputs in submissions/dbS_proba_va.csv and dbS_proba_te.csv


In [99]:
# 3-19 Validation only: deberta averaged over 4 seeds (42 + 1, 2, 3), blended with roberta (4 seeds) and local
dbS_va = (pd.read_csv("submissions/dbS_proba_va.csv", dtype={"post_id": str})
            .set_index("post_id").loc[va["post_id"], labels].values)
db4_va = (db8_va + 3 * dbS_va) / 4                                           # equal weight per seed
for name, P in [("rb4 .4 + db8 .2 + local .4 (submitted)", 0.4 * rb4_va + 0.2 * db8_va + 0.4 * P_local),
                ("rb4 .4 + db4 .2 + local .4",             0.4 * rb4_va + 0.2 * db4_va + 0.4 * P_local),
                ("rb4 .35 + db4 .3 + local .35",           0.35 * rb4_va + 0.3 * db4_va + 0.35 * P_local),
                ("rb4 + db4 + local, equal 1/3",           (rb4_va + db4_va + P_local) / 3)]:
    p = np.array(labels)[P.argmax(1)]
    print(f"{name:40s} macro F1={f1_score(va['label'], p, average='macro'):.4f}  acc={accuracy_score(va['label'], p):.4f}")

rb4 .4 + db8 .2 + local .4 (submitted)   macro F1=0.6434  acc=0.6667
rb4 .4 + db4 .2 + local .4               macro F1=0.6421  acc=0.6667
rb4 .35 + db4 .3 + local .35             macro F1=0.6474  acc=0.6726
rb4 + db4 + local, equal 1/3             macro F1=0.6487  acc=0.6741


In [100]:
# 3-20 Test: rb4 + db4 + local, equal 1/3 (val macro F1 0.6487), write the submission file
dbS_te = (pd.read_csv("submissions/dbS_proba_te.csv", dtype={"post_id": str})
            .set_index("post_id").loc[test["post_id"], labels].values)
db4_te = (db8_te + 3 * dbS_te) / 4                                   # equal weight per seed
P_te7 = (rb4_te + db4_te + (0.5 * P_ft_te + 0.5 * proba_te)) / 3     # equal 1/3 blend
out = pd.DataFrame({"post_id": test["post_id"].values, "label": np.array(labels)[P_te7.argmax(1)]})
out = sub[["post_id"]].merge(out, on="post_id", how="left")
assert len(out) == 1440 and out["label"].notna().all()
out.rename(columns={"post_id": "id"}).to_csv("submissions/k3_rb4_db4_blend.csv", index=False)
print(out["label"].value_counts().to_dict())

{'joy': 356, 'anger': 347, 'sadness': 266, 'surprise': 195, 'disgust': 162, 'fear': 114}
